# Step 0. Dataset profile: the five BPJS claim tables

Describes the data as delivered, for the Data section of the paper: rows, columns, members, keys, period, missing values per column, and TB codes per table. Independent of the network steps.

Reads the tables through `common.load_tables()` (same cache as script 01; `harmonise=False` = values as delivered). Column labels come from `BPJS_variable_dictionary.csv` if that file is in `pipeline/` (export it from the project docs); without it the labels column stays empty.
Writes to `two_relations/dataset/`: `T_tables.csv/.tex`, `T_columns.csv`, `T_tb_codes.csv/.tex`, `T_visits_per_year.csv`. No patient-level values are written.

| Step | Notebook | Code file | Output folder (`graph_TB_ieee/results/two_relations/`) |
|---|---|---|---|
| 0 | `net_00_dataset.ipynb` | `common.py` | `dataset/` |
| 1 | `net_01_build.ipynb` | `net_common.py` | `networks/` |
| 2 | `net_02_louvain.ipynb` | `net_louvain.py` | `louvain/` |
| 3 | `net_03_leiden.ipynb` | `net_leiden.py` | `leiden/` |
| 4 | `net_04_label_propagation.ipynb` | `net_label_propagation.py` | `label_propagation/` |
| 5 | `net_05_girvan_newman.ipynb` | `net_girvan_newman.py` | `girvan_newman/` |
| 6 | `net_06_compare.ipynb` | `net_common.py` | `compare/` |
| 7 | `net_07_figures.ipynb` | – | `figs/<algorithm>/` |
| 8 | `net_08_runtime.ipynb` | – | `runtime/` |

Step 0 (dataset profile) is independent. Step 1 runs once. Steps 2–5 are independent: run them in any order, each one reads the networks from step 1. Every build and algorithm run appends its timing to `runtime/runtime_log.csv`; step 8 turns it into the runtime table.

## 0. Setup (Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/2025_doktor_ITS/COLAB/PAPER 1 IEEE SNA/bpjs-tb-graph/pipeline
%pip install -q python-igraph

In [2]:
import os, re
import numpy as np, pandas as pd
import common
from net_common import Config, algo_dir

cfg = Config()
OUT = algo_dir(cfg, "dataset")
try:
    T = common.load_tables(harmonise=False)       # values as delivered
except Exception as e:
    print(f"harmonise=False failed ({e}); using harmonised tables")
    T = common.load_tables(harmonise=True)
# common.load_tables() may also return non-table entries (e.g. an info dict): keep tables only
print("not tables (ignored):", [k for k, v in T.items() if not isinstance(v, pd.DataFrame)])
T = {k: v for k, v in T.items() if isinstance(v, pd.DataFrame)}
print({k: v.shape for k, v in T.items()})

(runtime is logged only when run as: !python <script>.py)
not tables (ignored): ['prov_map']
{'member': (118747, 21), 'fktp': (380429, 33), 'noncap': (17393, 25), 'fkrtl': (235671, 65), 'secdx': (294733, 6)}


## 1. Table overview (paper Table: data sources)

In [3]:
META = [  # key, name in codebook, grain, primary key, date columns
    ("member", "Membership (Kepesertaan)", "member",         ["PSTV01"],          []),
    ("fktp",   "FKTP capitation visits",   "clinic visit",   ["FKP02"],           ["FKP03"]),
    ("noncap", "FKTP non-capitation",      "service line",   ["PNK02", "PNK16"],  ["PNK03"]),
    ("fkrtl",  "FKRTL hospital visits",    "hospital visit", ["FKL02"],           ["FKL03", "FKL04"]),
    ("secdx",  "FKRTL secondary diagnoses","visit x code",   ["FKL02", "FKL24"],  []),
]
CODEBOOK_COL = re.compile(r"^(PSTV|FKP|FKL|PNK)\d+[A-Z]?$")        # in the BPJS codebook
DELIVERED_EXTRA = re.compile(r"^(PSTV|FKP|FKL|PNK)\d+[A-Z]?_NEW$")  # in the files, not in the codebook:
                                                                     # province after the 2022 Papua split
def origin(c):
    if CODEBOOK_COL.match(c): return "codebook"
    if DELIVERED_EXTRA.match(c): return "delivered, not in codebook"
    return "derived (common.py)"

def upper(df):
    df = df.copy(); df.columns = [str(c).strip().upper() for c in df.columns]; return df

def to_date(s):
    d = pd.to_datetime(s, errors="coerce", format="ISO8601")
    return d if d.notna().mean() > 0.5 else pd.to_datetime(s, errors="coerce", dayfirst=True)

D = {k: upper(T[k]) for k, *_ in META if k in T and T[k] is not None}
fkrtl_member = D["fkrtl"].drop_duplicates("FKL02").set_index("FKL02")["PSTV01"] if "fkrtl" in D else None
rows = []
for k, name, grain, pk, dates in META:
    if k not in D:
        continue
    df = D[k]
    org = pd.Series([origin(c) for c in df.columns])
    if "PSTV01" in df:
        members = df["PSTV01"].nunique()
    elif "FKL02" in df and fkrtl_member is not None:
        members = df["FKL02"].map(fkrtl_member).nunique()
    else:
        members = np.nan
    pkc = [c for c in pk if c in df]
    dup = int(df.duplicated(pkc, keep=False).sum()) if pkc else np.nan
    period = "-"
    if dates:
        dd = pd.concat([to_date(df[c]) for c in dates if c in df]).dropna()
        period = f"{dd.min():%Y-%m-%d} to {dd.max():%Y-%m-%d}" if len(dd) else "-"
    rows.append({"Table": name, "Grain": grain, "Key": "+".join(pk), "Rows": len(df),
                 "Columns delivered": int((org != "derived (common.py)").sum()),
                 "of which codebook": int((org == "codebook").sum()),
                 "of which _NEW (not in codebook)": int((org == "delivered, not in codebook").sum()),
                 "Derived columns (not counted)": int((org == "derived (common.py)").sum()),
                 "Members": members, "Rows with duplicate key": dup, "Period": period})
tables = pd.DataFrame(rows)
tables.to_csv(f"{OUT}/T_tables.csv", index=False)
try:
    tables.to_latex(f"{OUT}/T_tables.tex", index=False, escape=True, float_format="%.0f",
                    caption="BPJS Kesehatan sample: source tables", label="tab:tables")
except Exception as e:
    print("LaTeX export skipped:", e)
tables

,Table,Grain,Key,Rows,Columns delivered,of which codebook,of which _NEW (not in codebook),Derived columns (not counted),Members,Rows with duplicate key,Period
0,Membership (Kepesertaan),member,PSTV01,118747,20,18,2,1,118747,0,-
1,FKTP capitation visits,clinic visit,FKP02,380429,28,26,2,5,59590,0,2024-01-01 to 2024-12-31
2,FKTP non-capitation,service line,PNK02+PNK16,17393,24,23,1,1,4500,0,2023-12-27 to 2024-12-31
3,FKRTL hospital visits,hospital visit,FKL02,235671,57,55,2,8,31655,0,2023-11-16 to 2024-12-31
4,FKRTL secondary diagnoses,visit x code,FKL02+FKL24,294733,4,4,0,2,27006,0,-


`Rows with duplicate key`: 0 means the key is unique (FKTP non-capitation is checked on PNK02 + PNK16 because one visit has several service lines).
**Columns delivered** = codebook columns + `*_NEW` province columns that BPJS ships but the codebook does not list (38 provinces after the 2022 Papua split). **Derived columns** are made by `common.py` during cleaning (parsed dates, `DX3`, `PROVINCE | DISTRICT` keys, cost, source file) and are not data columns.

## 2. Column dictionary: type, missing %, distinct values

In [4]:
# BPJS_variable_dictionary.csv: searched in pipeline/, in DATA_DIR from config.py, and below
DICT_PATHS = ["BPJS_variable_dictionary.csv",
              "/content/drive/MyDrive/2025_doktor_ITS/DATASET/BPJS_TB/BPJS_variable_dictionary.csv"]
try:
    import config
    DICT_PATHS.insert(1, os.path.join(str(config.DATA_DIR), "BPJS_variable_dictionary.csv"))
except Exception:
    pass
import glob
DICT_PATHS = ["codebook/BPJS_variable_dictionary.csv", *DICT_PATHS]
# also any file named like *BPJS_variable_dictionary*.csv inside pipeline/ or the repo
DICT_PATHS += sorted(glob.glob("**/*BPJS_variable_dictionary*.csv", recursive=True)
                     + glob.glob("../**/*BPJS_variable_dictionary*.csv", recursive=True))
dic = None
for p in DICT_PATHS:
    if os.path.exists(p):
        dic = pd.read_csv(p, encoding="utf-8-sig")
        print("dictionary:", p, dic.shape)
        break
if dic is None:
    print("WARNING: BPJS_variable_dictionary.csv not found in", DICT_PATHS, "-> label_en stays empty")
cols = []
for k, name, *_ in META:
    if k not in D:
        continue
    df = D[k]
    for c in df.columns:
        s = df[c]
        miss = s.isna() | s.astype("string").str.strip().isin(["", "nan", "NaN", "NaT"])
        lab = None
        if dic is not None:
            hit = dic[(dic["table"] == name) & (dic["variable"] == c)]
            lab = hit["label_en"].iat[0] if len(hit) else None
        cols.append({"table": name, "column": c, "label_en": lab,
                     "origin": origin(c), "dtype": str(s.dtype),
                     "missing_pct": round(100 * miss.mean(), 2), "distinct": int(s.nunique(dropna=True))})
columns = pd.DataFrame(cols)
columns.to_csv(f"{OUT}/T_columns.csv", index=False)
print(pd.crosstab(columns["table"], columns["origin"], margins=True))
columns.sort_values(["table", "missing_pct"], ascending=[True, False]).groupby("table").head(8)

dictionary: /content/drive/MyDrive/2025_doktor_ITS/DATASET/BPJS_TB/BPJS_variable_dictionary.csv (126, 7)
origin                     codebook  delivered, not in codebook  derived (common.py)  All
table                                                                                    
FKRTL hospital visits            55                           2                    8   65
FKRTL secondary diagnoses         4                           0                    2    6
FKTP capitation visits           26                           2                    5   33
FKTP non-capitation              23                           1                    1   25
Membership (Kepesertaan)         18                           2                    1   21
All                             126                           7                   17  150


,table,column,label_en,origin,dtype,missing_pct,distinct
126,FKRTL hospital visits,FKL39,Special prosthesis description,codebook,object,99.99,2
132,FKRTL hospital visits,FKL45,Special drug description,codebook,object,99.93,7
129,FKRTL hospital visits,FKL42,Special investigation description,codebook,object,99.91,1
123,FKRTL hospital visits,FKL36,Special procedure description,codebook,object,99.73,10
117,FKRTL hospital visits,FKL30,Procedure type,codebook,object,49.27,12645
82,FKRTL hospital visits,FKP02,Originating FKTP visit ID,codebook,object,38.15,48862
118,FKRTL hospital visits,FKL31,Tariff region,codebook,object,0.18,5
79,FKRTL hospital visits,PSTV01,Member ID,codebook,object,0.00,31655
144,FKRTL secondary diagnoses,FKL02,Hospital visit ID,codebook,object,0.00,199332
145,FKRTL secondary diagnoses,FKL24,"Secondary diagnosis, full",codebook,object,0.00,4646


## 3. TB codes (A15–A19) per table

In [5]:
TB = ("A15", "A16", "A17", "A18", "A19")
DX = {"fktp": ["FKP14A"], "noncap": ["PNK13A"], "fkrtl": ["FKL15A", "FKL17A"], "secdx": ["FKL24A"]}
def icd3(s): return s.astype("string").str.upper().str.replace(r"[^A-Z0-9]", "", regex=True).str[:3]
tb_rows, tb_ids = [], set()
for k, name, *_ in META:
    if k not in DX or k not in D:
        continue
    df = D[k]
    hit = pd.Series(False, index=df.index)
    for c in DX[k]:
        if c in df:
            hit |= icd3(df[c]).isin(TB)
    ids = (df.loc[hit, "PSTV01"] if "PSTV01" in df else df.loc[hit, "FKL02"].map(fkrtl_member)).dropna()
    tb_ids |= set(ids)
    tb_rows.append({"Table": name, "Diagnosis columns": ", ".join(DX[k]), "Rows": len(df),
                    "Rows with A15-A19": int(hit.sum()), "% rows": round(100 * hit.mean(), 2),
                    "Patients with A15-A19": ids.nunique()})
tb = pd.DataFrame(tb_rows)
tb.loc[len(tb)] = {"Table": "Any table (union)", "Patients with A15-A19": len(tb_ids)}
for c in ("Rows", "Rows with A15-A19", "Patients with A15-A19"):
    tb[c] = tb[c].astype("Int64")
tb.to_csv(f"{OUT}/T_tb_codes.csv", index=False)
try:
    tb.to_latex(f"{OUT}/T_tb_codes.tex", index=False, na_rep="", escape=True, float_format="%.1f",
                caption="Rows and patients with a TB code (A15--A19)", label="tab:tbcodes")
except Exception as e:
    print("LaTeX export skipped:", e)
tb

,Table,Diagnosis columns,Rows,Rows with A15-A19,% rows,Patients with A15-A19
0,FKTP capitation visits,FKP14A,380429,17518,4.60,7109
1,FKTP non-capitation,PNK13A,17393,149,0.86,129
2,FKRTL hospital visits,"FKL15A, FKL17A",235671,23157,9.83,7095
3,FKRTL secondary diagnoses,FKL24A,294733,22708,7.70,5500
4,Any table (union),NaN,<NA>,<NA>,NaN,10839


## 3b. TB codes by sample year (`PSTV16`)
Members enter the TB sample in the year of a TB diagnosis and are followed up, so the share with an A15–A19 code in 2024 depends on the entry year (2024 entrants ≈ all, 2022 entrants few).

In [6]:
m = D["member"]
yr_s = m["PSTV16"].astype("string").str[:4].rename("sample year (PSTV16)")
tbflag = m["PSTV01"].isin(tb_ids).map({True: "TB-coded 2024", False: "no TB code 2024"})
by_year = pd.crosstab(yr_s, tbflag, margins=True, margins_name="All")
by_year["% TB-coded 2024"] = (100 * by_year["TB-coded 2024"] / by_year["All"]).round(1)
by_year.to_csv(f"{OUT}/T_tb_by_sample_year.csv")
try:
    by_year.to_latex(f"{OUT}/T_tb_by_sample_year.tex", escape=True, float_format="%.1f",
                     caption="Members with an A15--A19 code in 2024 by sample year", label="tab:tbyear")
except Exception as e:
    print("LaTeX export skipped:", e)
by_year

PSTV01,TB-coded 2024,no TB code 2024,All,% TB-coded 2024
sample year (PSTV16),,,,
2022,3651,104927,108578,3.4
2023,2161,2955,5116,42.2
2024,5027,26,5053,99.5
All,10839,107908,118747,9.1


## 3c. Active TB, past TB (B90) and 2024 status by sample year
A15–A19 = active TB; **B90 = sequelae of tuberculosis** (late effects after TB, not active disease). Both are values of the same diagnosis columns. Members with neither are split by what their 2024 claims show.

In [7]:
TBC = ("A15", "A16", "A17", "A18", "A19")
def patients_with(codes):
    out = {}
    for k, cols in DX.items():
        df = D[k]
        ids_all = df["PSTV01"] if "PSTV01" in df else df["FKL02"].map(fkrtl_member)
        for c in cols:
            if c in df:
                hit = icd3(df[c]).isin(codes)
                out[(k, c)] = (int(hit.sum()), set(ids_all[hit].dropna()))
    return out
a, b = patients_with(TBC), patients_with(("B90",))
by_col = pd.DataFrame([{"table": k, "column": c,
                        "rows A15-A19": a[(k, c)][0], "patients A15-A19": len(a[(k, c)][1]),
                        "rows B90": b[(k, c)][0], "patients B90": len(b[(k, c)][1])} for (k, c) in a])
b_ids = set().union(*[v[1] for v in b.values()])
by_col.to_csv(f"{OUT}/T_tb_b90_by_column.csv", index=False)
print(f"B90 patients: {len(b_ids):,} | also A15-A19 in 2024: {len(b_ids & tb_ids):,} | B90 only: {len(b_ids - tb_ids):,}")
by_col

B90 patients: 3,123 | also A15-A19 in 2024: 1,802 | B90 only: 1,321


,table,column,rows A15-A19,patients A15-A19,rows B90,patients B90
0,fktp,FKP14A,17518,7109,1140,690
1,noncap,PNK13A,149,129,19,15
2,fkrtl,FKL15A,20659,5744,2857,726
3,fkrtl,FKL17A,4143,3265,1419,957
4,secdx,FKL24A,22708,5500,8220,2366


In [8]:
m = D["member"]
visited = set(D["fktp"]["PSTV01"]) | set(D["fkrtl"]["PSTV01"]) | set(D["noncap"]["PSTV01"])
dx_any = set()
for k, cols in DX.items():
    df = D[k]
    ids = df["PSTV01"] if "PSTV01" in df else df["FKL02"].map(fkrtl_member)
    for c in cols:
        if c in df:
            dx_any |= set(ids[icd3(df[c]).str.match(r"[A-Z]\d\d", na=False)].dropna())
died = pd.to_numeric(m["PSTV18"], errors="coerce")
pid = m["PSTV01"]
status = pd.Series(np.select(
    [pid.isin(tb_ids), pid.isin(b_ids), died.lt(2024), ~pid.isin(visited), pid.isin(dx_any)],
    ["1 active TB 2024 (A15-A19)", "2 past TB 2024 (B90 only)", "3 died before 2024",
     "4 alive, no visit in 2024", "5 visits, other diagnoses only"],
    "6 visits, no diagnosis (wellness only)"), index=m.index, name="status 2024")
st = pd.crosstab(status, m["PSTV16"].astype("string").str[:4].rename("sample year"), margins=True, margins_name="All")
pct = (100 * st.div(st.loc["All"], axis=1)).round(1)
st.to_csv(f"{OUT}/T_status_2024_by_sample_year.csv")
pct.to_csv(f"{OUT}/T_status_2024_by_sample_year_pct.csv")
try:
    st.to_latex(f"{OUT}/T_status_2024_by_sample_year.tex", escape=True,
                caption="2024 status of TB-sample members by sample year", label="tab:status2024")
except Exception as e:
    print("LaTeX export skipped:", e)
display(st)
pct

sample year,2022,2023,2024,All
status 2024,,,,
1 active TB 2024 (A15-A19),3651,2161,5027,10839
2 past TB 2024 (B90 only),1272,49,0,1321
3 died before 2024,13263,0,0,13263
"4 alive, no visit in 2024",41511,1368,7,42886
"5 visits, other diagnoses only",42747,1351,19,44117
"6 visits, no diagnosis (wellness only)",6134,187,0,6321
All,108578,5116,5053,118747


sample year,2022,2023,2024,All
status 2024,,,,
1 active TB 2024 (A15-A19),3.4,42.2,99.5,9.1
2 past TB 2024 (B90 only),1.2,1.0,0.0,1.1
3 died before 2024,12.2,0.0,0.0,11.2
"4 alive, no visit in 2024",38.2,26.7,0.1,36.1
"5 visits, other diagnoses only",39.4,26.4,0.4,37.2
"6 visits, no diagnosis (wellness only)",5.6,3.7,0.0,5.3
All,100.0,100.0,100.0,100.0


## 4. Visits per year

In [9]:
yr = {}
for k, col, lab in (("fktp", "FKP03", "FKTP visits (FKP03)"), ("noncap", "PNK03", "Non-capitation lines (PNK03)"),
                    ("fkrtl", "FKL03", "FKRTL admissions (FKL03)")):
    if k in D and col in D[k]:
        yr[lab] = to_date(D[k][col]).dt.year.value_counts()
yr = pd.DataFrame(yr).fillna(0).astype(int).sort_index()
yr.index = yr.index.astype(int).rename("year")
yr.to_csv(f"{OUT}/T_visits_per_year.csv")
print("saved to", os.path.abspath(OUT))
yr

saved to /content/drive/MyDrive/2025_doktor_ITS/graph_TB_ieee/results/two_relations/dataset


,FKTP visits (FKP03),Non-capitation lines (PNK03),FKRTL admissions (FKL03)
year,,,
2023,0,5,219
2024,380429,17388,235452
